# **-Dependencias**

-Montar Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


-Instalar AirFlow

In [2]:
import os
import sys

python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
constraint_url = f"https://raw.githubusercontent.com/apache/airflow/constraints-2.11.2/constraints-{python_version}.txt"
!pip install 'apache-airflow==2.11.2' 'great-expectations>=1.0.0' \
    --constraint {constraint_url} > /dev/null 2>&1

os.environ["AIRFLOW_HOME"] = "/content/airflow"
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
os.environ["AIRFLOW__WEBSERVER__WEB_SERVER_PORT"] = "8081"

!airflow db migrate > /dev/null 2>&1
!mkdir -p /content/airflow/dags

print("AirFlow instalado y configurado")

AirFlow instalado y configurado


-Instalar dash

In [3]:
!pip install dash dash-bootstrap-components --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 6.8 MB/s eta 0:00:00


-Instalar Kafka

In [4]:
# Descargar Kafka desde el archivo oficial de Apache
!wget -q https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz

# Configurar el almacenamiento usando KRaft (sin Zookeeper)
!uuid=$(./kafka_2.13-3.7.0/bin/kafka-storage.sh random-uuid) && ./kafka_2.13-3.7.0/bin/kafka-storage.sh format -t $uuid -c ./kafka_2.13-3.7.0/config/kraft/server.properties

# Iniciar el servidor de Kafka en modo "daemon" (segundo plano)
!./kafka_2.13-3.7.0/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.7.0/config/kraft/server.properties

# Instalar la librería oficial de Confluent para Python
!pip install -q confluent-kafka

# Damos unos segundos para asegurar que el broker esté arriba y escuchando
import time
print("Iniciando Broker de Kafka local...")
time.sleep(10)
print("¡Entorno listo!")

metaPropertiesEnsemble=MetaPropertiesEnsemble(metadataLogDir=Optional.empty, dirs={/tmp/kraft-combined-logs: EMPTY})
Formatting /tmp/kraft-combined-logs with metadata.version 3.7-IV4.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 25.0 MB/s eta 0:00:00
Iniciando Broker de Kafka local...
¡Entorno listo!


Autenticacion en BigQuery
Se autentica antes de iniciar Airflow para que los tasks hereden las credenciales ADC (application default credentials)

In [5]:
from google.colab import auth, userdata
auth.authenticate_user()

#exportar project_id para que las tareas del DAG lo lean como variable de entorno
os.environ["GCP_PROJECT_ID"] = userdata.get("project_gcp")
print("BigQuery autenticado")

BigQuery autenticado


# **-Fuentes de Datos:**

1) Base de datos: El DWH (SQlite) desarrollado a lo largo de las anteriores entregas.
2) Google BigQuery: Conexión al proyecto bigquery-public-data, dataset covid19_jhu_csse, tabla deaths (solo se extraen datos de Ecuador)

# **-DAG**

In [6]:
%%writefile /content/airflow/dags/etl_DAG_ecuador_final_delivery.py
from airflow import DAG #Directed Acyclic Graph
from airflow.operators.python import PythonOperator #importa el operador que permite ejecutar funciones Python dentro de un DAG
from datetime import datetime

default_args = {
    "owner": "data_engineer",
    "depends_on_past": False, # La tarea no depende de ejecuciones anteriores
    "retries": 3,  #3 reintentos atomaticos si falla una task
}

with DAG(
    "etl_final_delivery",   #nombre delDAG
    default_args      = default_args,    #usa la config por defecto (la de arriba)
    schedule_interval = "@daily",           #se ejecuta diariamenre
    start_date        = datetime(2018, 1, 1),    #
    catchup           = False,    #no ejecuta las fechas pendientes pasadas desde el start date hasta el presente
) as dag:


  # TASK 1 -> Extraccion desde BD SQlite

  def extract_db():
        import sqlite3
        import pandas as pd

        ruta_dwh_second_delivery = "/content/drive/Shared drives/ETL_second_delivery/vacunacion_ecuador_dw_second_delivery.db"

        conn = sqlite3.connect(ruta_dwh_second_delivery)
        pd.read_sql("SELECT * FROM fact_vacunacion", conn).to_csv("/content/tmp_fact.csv",      index=False)
        pd.read_sql("SELECT * FROM dim_fecha",       conn).to_csv("/content/tmp_fecha.csv",     index=False)
        pd.read_sql("SELECT * FROM dim_canton",      conn).to_csv("/content/tmp_canton.csv",    index=False)
        pd.read_sql("SELECT * FROM dim_provincia",   conn).to_csv("/content/tmp_provincia.csv", index=False)
        pd.read_sql("SELECT * FROM dim_region",      conn).to_csv("/content/tmp_region.csv",    index=False)
        pd.read_sql("SELECT * FROM dim_indice_uhc",  conn).to_csv("/content/tmp_indice_uhc.csv",    index=False)
        conn.close()
        print("Extracion de BD finalizada")


  # TASK 2 -> Extraccion desde BQ (muertes COVID-19 Ecuador)

  def extract_bigquery():
      import os
      import pandas as pd
      from google.cloud import bigquery

      client = bigquery.Client(project=os.environ.get("GCP_PROJECT_ID"))

      query = '''
          SELECT *
          FROM `bigquery-public-data.covid19_jhu_csse.deaths`
          WHERE country_or_region = "Ecuador"
      '''

      df = client.query(query).to_dataframe()
      df.to_csv("/content/tmp_covid_raw.csv", index=False)
      print(f"Extraccion de BigQuery completada - {df.shape[0]} filas, {df.shape[1]} columnas")


  # TASK 3 -> Transformaciones y Merge

  def transform():
        import pandas as pd

        #Fuente 1:
        df_fact       = pd.read_csv("/content/tmp_fact.csv")
        df_dim_fecha  = pd.read_csv("/content/tmp_fecha.csv")
        df_dim_canton = pd.read_csv("/content/tmp_canton.csv")
        df_dim_prov   = pd.read_csv("/content/tmp_provincia.csv")
        df_dim_reg    = pd.read_csv("/content/tmp_region.csv")
        df_dim_indice = pd.read_csv("/content/tmp_indice_uhc.csv")



        #Limpiar dim_fecha
        df_dim_fecha["fecha"] = pd.to_datetime(df_dim_fecha["fecha"], errors="coerce")  #convierte a datetime

        for col in ["anio","mes","trimestre","semana","dia","dia_semana"]:
            df_dim_fecha[col] = df_dim_fecha[col].astype(int)  #convierte en entero

        duplicados_identicos = df_dim_fecha[df_dim_fecha.duplicated(keep=False)] #ver los duplicados identicos para decidir si eliminarlos es correcto
        print("Duplicados identicos:")
        print(duplicados_identicos)
        df_dim_fecha = df_dim_fecha.drop_duplicates()

        duplicados_por_id = df_dim_fecha[df_dim_fecha.duplicated(subset="id", keep=False)] #ver duplicados por id para decidir si eliminarlos es correcto
        print("\nDuplicados por ID:")                                                      #(asegurar integridad referencial por id)
        print(duplicados_por_id.sort_values("id"))
        if not duplicados_por_id.empty:
          raise ValueError("Hay IDs duplicados. Revisar antes de continuar.")
        #df_dim_fecha = df_dim_fecha.drop_duplicates(subset="id")


        #Limpiar dim_canton
        df_dim_canton["canton_nombre"]    = df_dim_canton["canton_nombre"].str.strip().str.title()  #eliminar espacios y poner primera en mayuscula
        df_dim_canton["canton_poblacion"] = pd.to_numeric(df_dim_canton["canton_poblacion"], errors="coerce").fillna(0).astype(int)
        df_dim_canton = df_dim_canton.drop_duplicates()

        duplicados_por_id = df_dim_canton[df_dim_canton.duplicated(subset="canton_id", keep=False)]
        print("Duplicados por canton_id:")
        print(duplicados_por_id)
        if not duplicados_por_id.empty:
            raise ValueError("Hay canton_id duplicados. Revisar.")
        #df_dim_canton = df_dim_canton.drop_duplicates(subset="canton_id")


        #Limpiar dim_provincia
        df_dim_prov["provincia_nombre"]    = df_dim_prov["provincia_nombre"].str.strip().str.title()
        df_dim_prov["provincia_poblacion"] = pd.to_numeric(df_dim_prov["provincia_poblacion"], errors="coerce").fillna(0).astype(int) #convertir a entero
        df_dim_prov = df_dim_prov.drop_duplicates() #eliminar duplicados exactos

        duplicados_por_id = df_dim_prov[df_dim_prov.duplicated(subset="id", keep=False)]
        print("Duplicados por id:")
        print(duplicados_por_id)
        if not duplicados_por_id.empty:
            raise ValueError("Hay ids duplicados. Revisar.")
        #df_dim_prov = df_dim_prov.drop_duplicates(subset="id")


        #Limpiar dim_region
        df_dim_reg["region_nombre"] = df_dim_reg["region_nombre"].str.strip().str.title()
        df_dim_reg["zona"] = pd.to_numeric(df_dim_reg["zona"], errors="coerce").fillna(0).astype(int)
        df_dim_reg = df_dim_reg.drop_duplicates()

        duplicados_por_id = df_dim_reg[df_dim_reg.duplicated(subset = "id", keep = False)]
        print("Duplicados por id:")
        print(duplicados_por_id)
        if not duplicados_por_id.empty:
          raise ValueError("Hay ids duplicados. Revisar.")
        #df_dim_reg = df_dim_reg.drop_duplicates(subset="id")


        #Limpiar fact_vacunacion
        for col in ["dosis_total","primera_dosis","segunda_dosis","dosis_unica",
                    "dosis_refuerzo","poblacion_canton","poblacion_provincia"]:
            df_fact[col] = pd.to_numeric(df_fact[col], errors="coerce").fillna(0).astype(int)
        df_fact = df_fact.drop_duplicates()

        duplicados_por_id = df_fact[df_fact.duplicated(subset = "id", keep = False)]
        print("Duplicados por id:")
        print(duplicados_por_id)
        if not duplicados_por_id.empty:
          raise ValueError("Hay ids duplicados. Revisar.")

        #limpiar dim indice_uhc
        df_dim_indice["uhc_score"] = pd.to_numeric(df_dim_indice["uhc_score"].astype(str).str.strip(),errors="coerce")
        df_dim_indice = df_dim_indice.drop_duplicates()

        duplicados_por_anio = df_dim_indice[df_dim_indice.duplicated(subset = "anio", keep = False)]
        print("Duplicados por anio:")
        print(duplicados_por_anio)
        if not duplicados_por_anio.empty:
          raise ValueError("Hay anios duplicados. Revisar.")


        #Fuente 2: COVID deaths (wide -> long)

        df_covid_raw = pd.read_csv("/content/tmp_covid_raw.csv")

        meta_cols = ["province_or_state","country_or_region","latitude","longitude","location_geom"]
        date_cols = [c for c in df_covid_raw.columns if c not in meta_cols]

        df_covid = df_covid_raw[["country_or_region"] + date_cols].melt(
            id_vars=["country_or_region"],
            var_name="fecha_raw",
            value_name="muertes_acumuladas")

        def parse_covid_date(col):  #convierte a formato fecha
            try:
                parts = col.strip("_").split("_")  #elimina los _ presentes en la fecha y divide la fecha usando el _
                return pd.to_datetime(f"20{parts[2]}-{parts[0].zfill(2)}-{parts[1].zfill(2)}")  #cambia el formayo a año/mes/dia
            except Exception:
                return pd.NaT  #si falla devuelve not a time

        df_covid["fecha"] = df_covid["fecha_raw"].apply(parse_covid_date)
        df_covid["muertes_acumuladas"] = pd.to_numeric(df_covid["muertes_acumuladas"], errors="coerce").fillna(0).astype(int)
        df_covid = df_covid.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)  #si hay nat los elimina
        df_covid["muertes_diarias"] = df_covid["muertes_acumuladas"].diff().clip(lower=0).fillna(0).astype(int)  #calcula las muertes diarias (calcula la diferencia entre cada fila y la anterior)
        df_covid["anio"] = df_covid["fecha"].dt.year
        df_covid["mes"] = df_covid["fecha"].dt.month
        df_covid["fecha"] = df_covid["fecha"].dt.strftime("%Y-%m-%d") #transforma de datetime a string
        df_covid = df_covid[["fecha","anio","mes","muertes_acumuladas","muertes_diarias"]]

        # Guardar CSVs transformados
        df_dim_fecha["fecha"] = df_dim_fecha["fecha"].dt.strftime("%Y-%m-%d")

        df_fact.to_csv("/content/tf_fact.csv",            index=False)
        df_dim_fecha.to_csv("/content/tf_fecha.csv",      index=False)
        df_dim_canton.to_csv("/content/tf_canton.csv",    index=False)
        df_dim_prov.to_csv("/content/tf_provincia.csv",   index=False)
        df_dim_reg.to_csv("/content/tf_region.csv",       index=False)
        df_dim_indice.to_csv("/content/tf_indice_uhc.csv",index=False)
        df_covid.to_csv("/content/tf_covid.csv",          index=False)

        print(f"Transformacion completada")



  # TASK 4 — Quality Checks (Great Expectations)
  def quality_checks():

        import great_expectations as gx
        import great_expectations.expectations as gxe
        import pandas as pd
        import warnings
        warnings.filterwarnings("ignore")

        print("Iniciando quality checks con Great Expectations")

        context = gx.get_context()
        errores = []

        def validar(nombre_tabla, df, reglas):
            batch = context.data_sources.pandas_default.read_dataframe(df)
            print(f"\n── {nombre_tabla} ──")
            for regla in reglas:
                resultado = batch.validate(regla)
                tipo      = regla.expectation_type
                columna   = getattr(regla, "column", "tabla")
                if resultado.success:
                    print(f"  ok  {columna} , {tipo}")
                else:
                    detalle = resultado.result.get("partial_unexpected_list", "sin detalle")
                    msg     = f"fallo [{nombre_tabla}] {columna} , {tipo} -> valores inesperados: {detalle}"
                    print(f"  {msg}")
                    errores.append(msg)

        # fact_vacunacion
        validar("fact_vacunacion", pd.read_csv("/content/tf_fact.csv"), [
            gxe.ExpectTableRowCountToBeBetween(min_value=1),
            gxe.ExpectColumnValuesToNotBeNull(column="date_id"),
            gxe.ExpectColumnValuesToNotBeNull(column="canton_id"),
            gxe.ExpectColumnValuesToNotBeNull(column="provincia_id"),
            gxe.ExpectColumnValuesToBeBetween(column="dosis_total",    min_value=0),
            gxe.ExpectColumnValuesToBeBetween(column="primera_dosis",  min_value=0),
            gxe.ExpectColumnValuesToBeBetween(column="segunda_dosis",  min_value=0),
            gxe.ExpectColumnValuesToBeBetween(column="dosis_unica",    min_value=0),
            gxe.ExpectColumnValuesToBeBetween(column="dosis_refuerzo", min_value=0),
        ])

        # dim_canton
        validar("dim_canton", pd.read_csv("/content/tf_canton.csv"), [
            gxe.ExpectTableRowCountToBeBetween(min_value=1),
            gxe.ExpectColumnValuesToNotBeNull(column="canton_id"),
            gxe.ExpectColumnValuesToBeUnique(column="canton_id"),
            gxe.ExpectColumnValuesToNotBeNull(column="canton_nombre"),
            gxe.ExpectColumnValuesToBeBetween(column="canton_poblacion", min_value=0),
        ])

        # dim_fecha
        validar("dim_fecha", pd.read_csv("/content/tf_fecha.csv"), [
            gxe.ExpectTableRowCountToBeBetween(min_value=1),
            gxe.ExpectColumnValuesToNotBeNull(column="id"),
            gxe.ExpectColumnValuesToBeUnique(column="id"),
            gxe.ExpectColumnValuesToNotBeNull(column="fecha"),
            gxe.ExpectColumnValuesToBeBetween(column="mes", min_value=1,  max_value=12),
            gxe.ExpectColumnValuesToBeBetween(column="dia", min_value=1,  max_value=31),
        ])

        # dim_indice_uhc
        validar("dim_indice_uhc", pd.read_csv("/content/tf_indice_uhc.csv"), [
            gxe.ExpectTableRowCountToBeBetween(min_value=1),
            gxe.ExpectColumnValuesToNotBeNull(column="anio"),
            gxe.ExpectColumnValuesToBeUnique(column="anio"),
            gxe.ExpectColumnValuesToNotBeNull(column="uhc_score"),
            gxe.ExpectColumnValuesToBeBetween(column="uhc_score", min_value=0, max_value=100),
        ])

        # covid_deaths_ecuador
        validar("covid_deaths", pd.read_csv("/content/tf_covid.csv"), [
            gxe.ExpectTableRowCountToBeBetween(min_value=1),
            gxe.ExpectColumnValuesToNotBeNull(column="fecha"),
            gxe.ExpectColumnValuesToBeBetween(column="muertes_acumuladas", min_value=0),
            gxe.ExpectColumnValuesToBeBetween(column="muertes_diarias",    min_value=0),
            gxe.ExpectColumnValuesToBeBetween(column="mes", min_value=1, max_value=12),
        ])

        # Resultado final
        if errores:
            raise Exception(
                f"Quality checks fallidos — {len(errores)} error(es):\n" +
                "\n".join(errores)
            )

        print("\n Quality Checks completados - todas las expectativas se cumplen")


  # TASK 5 — Carga al DWH final (SQLite)

  def load():
        import sqlite3
        import pandas as pd

        ruta_dwh_final_delivery = "/content/drive/Shared drives/ETL_final_delivery/dwh_final_delivery.db"

        conn = sqlite3.connect(ruta_dwh_final_delivery)
        cur  = conn.cursor()
        cur.execute("PRAGMA foreign_keys = ON;")

        cur.executescript("""
            DROP TABLE IF EXISTS fact_vacunacion;
            DROP TABLE IF EXISTS fact_decesos;
            DROP TABLE IF EXISTS dim_indice_uhc;
            DROP TABLE IF EXISTS dim_fecha;
            DROP TABLE IF EXISTS dim_canton;
            DROP TABLE IF EXISTS dim_provincia;
            DROP TABLE IF EXISTS dim_region;
        """)
        conn.commit()

        cur.executescript("""

            CREATE TABLE dim_fecha (
                id          INTEGER PRIMARY KEY,
                created_at  TEXT,
                fecha       TEXT    NOT NULL UNIQUE,
                anio        INTEGER NOT NULL,
                mes         INTEGER NOT NULL,
                trimestre   INTEGER NOT NULL,
                semana      INTEGER NOT NULL,
                dia         INTEGER NOT NULL,
                dia_semana  INTEGER NOT NULL
            );

            CREATE TABLE dim_canton (
                canton_id        INTEGER PRIMARY KEY,
                canton_nombre    TEXT    NOT NULL,
                canton_poblacion INTEGER
            );

            CREATE TABLE dim_provincia (
                id                   INTEGER PRIMARY KEY,
                provincia_nombre     TEXT    NOT NULL UNIQUE,
                provincia_poblacion  INTEGER
            );

            CREATE TABLE dim_region (
                id            INTEGER PRIMARY KEY,
                region_nombre TEXT    NOT NULL UNIQUE,
                zona          INTEGER
            );

            CREATE TABLE dim_indice_uhc (
                anio       INTEGER PRIMARY KEY,
                uhc_score  REAL    NOT NULL
            );

            CREATE TABLE fact_vacunacion (
                id                  INTEGER PRIMARY KEY,
                date_id             INTEGER NOT NULL REFERENCES dim_fecha(id),
                canton_id           INTEGER NOT NULL REFERENCES dim_canton(canton_id),
                provincia_id        INTEGER NOT NULL REFERENCES dim_provincia(id),
                region_id           INTEGER NOT NULL REFERENCES dim_region(id),
                dosis_total         INTEGER,
                primera_dosis       INTEGER,
                segunda_dosis       INTEGER,
                dosis_unica         INTEGER,
                dosis_refuerzo      INTEGER,
                poblacion_canton    INTEGER,
                poblacion_provincia INTEGER
            );

            CREATE TABLE fact_decesos (
                id                  INTEGER PRIMARY KEY AUTOINCREMENT,
                date_id             INTEGER NOT NULL UNIQUE REFERENCES dim_fecha(id),
                muertes_acumuladas  INTEGER NOT NULL,
                muertes_diarias     INTEGER NOT NULL
            );

        """)
        conn.commit()

        # Cargar dimensiones (respetando integridad referencial)
        df_fecha = pd.read_csv("/content/tf_fecha.csv")
        df_fecha.to_sql("dim_fecha", conn, if_exists="append", index=False)
        pd.read_csv("/content/tf_canton.csv").to_sql("dim_canton",    conn, if_exists="append", index=False)
        pd.read_csv("/content/tf_provincia.csv").to_sql("dim_provincia", conn, if_exists="append", index=False)
        pd.read_csv("/content/tf_region.csv").to_sql("dim_region",    conn, if_exists="append", index=False)
        pd.read_csv("/content/tf_indice_uhc.csv").to_sql("dim_indice_uhc", conn, if_exists="append", index=False)

        # fact_decesos: solo fechas que existan en dim_fecha
        df_covid = pd.read_csv("/content/tf_covid.csv")
        fecha_a_id = df_fecha.set_index("fecha")["id"].to_dict()
        df_decesos = df_covid[df_covid["fecha"].isin(fecha_a_id)].copy()
        df_decesos["date_id"] = df_decesos["fecha"].map(fecha_a_id)
        df_decesos = df_decesos[["date_id","muertes_acumuladas","muertes_diarias"]]
        df_decesos = df_decesos.drop_duplicates(subset="date_id")
        df_decesos.to_sql("fact_decesos", conn, if_exists="append", index=False)

        # fact_vacunacion
        pd.read_csv("/content/tf_fact.csv").to_sql("fact_vacunacion", conn, if_exists="append", index=False)

        conn.commit()

        # Verificar rowcounts
        tablas = pd.read_sql('SELECT name FROM sqlite_master WHERE type="table"', conn)
        for t in tablas["name"]:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0, 0]
            print(f"  {t:<30} {n:>8,} filas")

        conn.close()
        print("Load OK — DWH guardado en:", ruta_dwh_final_delivery)


  #Producer con confluente-Kafka

  def kafka_producer():
    import sqlite3, json, time
    from confluent_kafka import Producer
    import random

    conf = {'bootstrap.servers': 'localhost:9092'}  #direccion del broker (servidor Kafka que recibe y distribuye mensajes) de kafka
    producer = Producer(conf)      #instancia el productor de kafka

    def reporte_entrega(err, msg): #verificar que el mensaje fue enviado correctamente (callback (función que se pasa como argumento a otra función para que se ejecute cuando ocurra un evento))
        if err is not None:
            print(f'Error al entregar mensaje: {err}')
        else:
            print(f'Mensaje entregado a {msg.topic()} [Partición {msg.partition()}]')

    #topicos
    topico_vacunacion = 'eventos-vacunacion'
    topico_decesos    = 'eventos-decesos'

    conn = sqlite3.connect("/content/drive/Shared drives/ETL_final_delivery/dwh_final_delivery.db")

    #Extraer cols y registros ANTES de cerrar la conexión
    cur_vac = conn.execute("SELECT * FROM fact_vacunacion")
    cols_vac = [d[0] for d in cur_vac.description]
    rows_vac = cur_vac.fetchall()

    cur_dec = conn.execute("SELECT * FROM fact_decesos")
    cols_dec = [d[0] for d in cur_dec.description]
    rows_dec = cur_dec.fetchall()

    conn.close()

    # Unir ambas listas etiquetando el tópico de cada fila
    mensajes = (
        [(topico_vacunacion, cols_vac, row) for row in rows_vac] +
        [(topico_decesos,    cols_dec, row) for row in rows_dec])
    #random.shuffle(mensajes) #mezclar aleatoriamente

    for topico, cols, row in mensajes: #Recorre todos los mensajes
      msg = dict(zip(cols, row))  #une columnas con valores y convierte en diccionario
      producer.produce(topico, value=json.dumps(msg).encode("utf-8"), callback=reporte_entrega)  #evia el mensaje (convierte a json y luego a bytes)
      producer.poll(0) #procesa callbacks y eventos pendientes
      time.sleep(0.001) #espera entre mensajes

    producer.flush() #fuerza el envio de todos los mensajes pendientes
    print(f"Producer finalizado — {len(rows_vac)} mensajes vacunación, {len(rows_dec)} mensajes decesos")


  # Definicion de tareas y logica de ejecucion

  t_extract_db  = PythonOperator(task_id="extract_db",       python_callable=extract_db)
  t_extract_bq  = PythonOperator(task_id="extract_bigquery", python_callable=extract_bigquery)
  t_transform   = PythonOperator(task_id="transform",        python_callable=transform)
  t_quality     = PythonOperator(task_id="quality_checks",   python_callable=quality_checks)
  t_load        = PythonOperator(task_id="load",             python_callable=load)
  t_kafka = PythonOperator(task_id="kafka_producer", python_callable=kafka_producer)

  # Las 2 extracciones van primero (en paralelo), luego transform -> quality -> load -> kafka
  [t_extract_db, t_extract_bq] >> t_transform >> t_quality >> t_load >> t_kafka


Writing /content/airflow/dags/etl_DAG_ecuador_final_delivery.py


-Crear usuario administrador

In [7]:
!airflow users create \
    --username admin \
    --password admin \
    --firstname Admin \
    --lastname User \
    --role Admin \
    --email admin@example.com

[2026-05-17T03:25:56.925+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.schemas
[2026-05-17T03:25:56.937+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.tables
[2026-05-17T03:25:56.939+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.types
[2026-05-17T03:25:56.942+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.constraints
[2026-05-17T03:25:56.942+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.defaults
[2026-05-17T03:25:56.944+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.comments
/usr/local/lib/python3.12/dist-packages/flask_limiter/extension.py:324 UserWarning: Using the in-memory storage for tracking rate limits as no storage was explicitly specified. This is not recommended for production use. See: https://flask-limiter.readthedocs.io#configuring-a-storage-backend for documentation about configuring the storage backend.
[2026-05-17T03:25:57.905+0000] {override.py:1528} INFO - Inserted Role:

-Iniciar AirFlow + tunel publico (Cloudflare)

In [14]:
import time, re, os

print("Iniciando Airflow...")

get_ipython().system_raw("airflow webserver -p 8081 > /content/web.log 2>&1 &")
get_ipython().system_raw("airflow scheduler > /content/scheduler.log 2>&1 &")

# Esperar hasta que el webserver responda (máx 120 seg)
import urllib.request
print("Esperando que Airflow arranque", end="")
for _ in range(24):
    time.sleep(5)
    try:
        urllib.request.urlopen("http://localhost:8081/health", timeout=3)
        print("    listo")
        break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\nAirflow tardó más de lo esperado — revisar /content/web.log")

# Crear túnel solo cuando el webserver ya responde
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
os.system("chmod +x cloudflared")

get_ipython().system_raw("./cloudflared tunnel --url http://localhost:8081 > /content/tunnel.log 2>&1 &")

time.sleep(8)  # darle tiempo al túnel de registrarse

with open("/content/tunnel.log") as f:
    log = f.read()

url = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)

if url:
    print("Airflow UI:", url[0])
    print("Usuario: admin  |  Contraseña: admin")
else:
    print("No se encontró URL")

Iniciando Airflow...
Esperando que Airflow arranque...    listo
Airflow UI: https://beauty-salvation-booth-broadcasting.trycloudflare.com
Usuario: admin  |  Contraseña: admin


# **-Kafka**

-Consumer

In [9]:
import json
from confluent_kafka import Consumer

# Configuración del consumidor
conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'grupo-etl-final-delivery',
    'auto.offset.reset': 'latest'}  #lee desde el inicio (earliest)

consumer = Consumer(conf)

# Suscripción a ambos tópicos
consumer.subscribe(['eventos-vacunacion', 'eventos-decesos'])

print("Escuchando mensajes en tiempo real...\n", flush=True)

intentos = 0
total_vacunacion = 0
total_decesos    = 0

try:
    while intentos < 9000:
        msg = consumer.poll(timeout=0.01)

        if msg is None:
            intentos += 1
            continue

        if msg.error():
            print(f"Error de consumidor: {msg.error()}", flush=True)
            continue

        # Decodificamos el mensaje
        datos = json.loads(msg.value().decode('utf-8'))
        topico = msg.topic()

        # Procesamiento según tópico
        if topico == 'eventos-vacunacion':
            total_vacunacion += 1
            print(
                f"[VACUNACIÓN] "
                f"ID: {datos['id']} | "
                f"Dosis total: {datos['dosis_total']} | "
                f"Primera: {datos['primera_dosis']} | "
                f"Segunda: {datos['segunda_dosis']} | "
                f"Refuerzo: {datos['dosis_refuerzo']}", flush=True)

        elif topico == 'eventos-decesos':
            total_decesos += 1
            print(
                f"[DECESOS]    "
                f"ID: {datos['id']} | "
                f"Fecha ID: {datos['date_id']} | "
                f"Acumuladas: {datos['muertes_acumuladas']} | "
                f"Diarias: {datos['muertes_diarias']}", flush=True)

        intentos = 0  # Reinicia si llegan mensajes

    print(f"\nFin del lote — {total_vacunacion} eventos vacunación, {total_decesos} eventos decesos.", flush=True)

except KeyboardInterrupt:
    print("\nConsumo interrumpido manualmente.", flush=True)

finally:
    consumer.close()
    print("Consumidor cerrado correctamente.", flush=True)

Escuchando mensajes en tiempo real...

Error de consumidor: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Subscribed topic not available: eventos-decesos: Broker: Unknown topic or partition"}
Error de consumidor: KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Subscribed topic not available: eventos-vacunacion: Broker: Unknown topic or partition"}

Consumo interrumpido manualmente.
Consumidor cerrado correctamente.


# **-Dashboard en tiempo real**

In [10]:
import shutil
shutil.copy(
    "/content/drive/Shared drives/ETL_final_delivery/dashboard_rt.py",
    "/content/dashboard_rt.py")
print("Archivo copiado.")

Archivo copiado.


In [13]:
import subprocess, threading, time, urllib.request
from google.colab.output import eval_js

# 1 — Lanzar Dash en background
dash_proc = subprocess.Popen(
    ["python", "/content/dashboard_rt.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,)

# 2 — Esperar a que Dash esté listo (hasta 30 seg)
print("Esperando que Dash arranque", end="")
for _ in range(15):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:8050", timeout=2)
        print(" ✓ listo")
        break
    except:
        print(".", end="", flush=True)
else:
    print("\n Dash tardó más de lo esperado. Verifica el proceso.")

# 3 — URL via proxy nativo de Colab (sin Cloudflare)
url = eval_js("google.colab.kernel.proxyPort(8050)")
print(f"\n Dashboard RealTime: {url}")

Esperando que Dash arranque ✓ listo

 Dashboard RealTime: https://8050-m-s-kkb-use1b1-3w3piq4a7chft-b.us-east1-1.prod.colab.dev


In [15]:
#parar grafico
#try:
#    dash_proc.terminate()
#    print("Dashboard detenido.")
#except NameError:
#    print("No había proceso activo.")

Dashboard detenido.
